# SHL Audio Grammar Scoring Challenge

Predicting grammar scores (0–5) from spoken audio using Whisper transcription, linguistic + acoustic features, and ensemble regression.

In [ ]:
!pip install -q openai-whisper librosa scikit-learn xgboost lightgbm nltk

In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
import librosa
import whisper
import nltk
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr

try:
    import xgboost as xgb
except ImportError:
    xgb = None
try:
    import lightgbm as lgb
except ImportError:
    lgb = None

warnings.filterwarnings("ignore")

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)
nltk.download("stopwords", quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk import pos_tag
from nltk.corpus import stopwords

STOP_WORDS = set(stopwords.words("english"))

In [ ]:
BASE_DIR = "/kaggle/input/shl-audio-scoring-challenge/dataset"
TRAIN_DIR = os.path.join(BASE_DIR, "audios", "train")
TEST_DIR = os.path.join(BASE_DIR, "audios", "test")
TRAIN_CSV = os.path.join(BASE_DIR, "csvs", "train.csv")
TEST_CSV = os.path.join(BASE_DIR, "csvs", "test.csv")

if not os.path.exists(TRAIN_CSV):
    for root, dirs, files in os.walk("/kaggle/input/shl-audio-scoring-challenge"):
        for f in files:
            if f == "train.csv":
                TRAIN_CSV = os.path.join(root, f)
                TEST_CSV = os.path.join(root, "test.csv")
                parent = os.path.dirname(root)
                for d in os.listdir(parent):
                    dp = os.path.join(parent, d)
                    if os.path.isdir(dp) and "audio" in d.lower():
                        for sub in os.listdir(dp):
                            sp = os.path.join(dp, sub)
                            if os.path.isdir(sp) and "train" in sub.lower():
                                TRAIN_DIR = sp
                            if os.path.isdir(sp) and "test" in sub.lower():
                                TEST_DIR = sp
                break
        if os.path.exists(TRAIN_CSV):
            break

print(f"Train CSV: {TRAIN_CSV} [{os.path.exists(TRAIN_CSV)}]")
print(f"Test CSV:  {TEST_CSV} [{os.path.exists(TEST_CSV)}]")
print(f"Train Dir: {TRAIN_DIR} [{os.path.exists(TRAIN_DIR)}]")
print(f"Test Dir:  {TEST_DIR} [{os.path.exists(TEST_DIR)}]")

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

fname_col = "filename"
label_col = "label"

print(f"Train: {train_df.shape}, Test: {test_df.shape}")
print(train_df.head())
print(f"\nLabel stats:\n{train_df[label_col].describe()}")

## Whisper Transcription

In [ ]:
whisper_model = whisper.load_model("base")

def transcribe_audio(filepath, model):
    try:
        result = model.transcribe(filepath, language="en", fp16=False)
        return result["text"].strip()
    except Exception as e:
        return ""

def get_audio_path(filename, audio_dir):
    path = os.path.join(audio_dir, filename)
    if os.path.exists(path):
        return path
    for ext in [".wav", ".mp3", ".flac"]:
        if os.path.exists(os.path.join(audio_dir, filename + ext)):
            return os.path.join(audio_dir, filename + ext)
    return path

print("Transcribing training audio...")
train_transcripts = []
for i, row in train_df.iterrows():
    fpath = get_audio_path(str(row[fname_col]), TRAIN_DIR)
    train_transcripts.append(transcribe_audio(fpath, whisper_model))
    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(train_df)}")
train_df["transcript"] = train_transcripts

print("Transcribing test audio...")
test_transcripts = []
for i, row in test_df.iterrows():
    fpath = get_audio_path(str(row[fname_col]), TEST_DIR)
    test_transcripts.append(transcribe_audio(fpath, whisper_model))
    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(test_df)}")
test_df["transcript"] = test_transcripts

print(f"Done. Train: {len(train_transcripts)}, Test: {len(test_transcripts)}")
for i in range(3):
    print(f"\n[{train_df[fname_col].iloc[i]}] score={train_df[label_col].iloc[i]}")
    print(f"  {train_df['transcript'].iloc[i][:150]}")

## Feature Extraction

In [ ]:
def extract_linguistic_features(text):
    features = {}
    keys = [
        "word_count", "unique_word_count", "char_count",
        "avg_word_length", "sentence_count", "avg_sentence_length",
        "type_token_ratio", "lexical_density", "stopword_ratio", "long_word_ratio",
        "noun_ratio", "verb_ratio", "adj_ratio", "adv_ratio",
        "prep_ratio", "conj_ratio", "pron_ratio",
        "pos_diversity", "word_length_std",
        "hapax_legomena_ratio", "avg_syllables_per_word",
        "complex_word_ratio", "words_per_minute_approx",
    ]
    if not text or len(text.strip()) == 0:
        return {k: 0 for k in keys}

    words = word_tokenize(text.lower())
    sentences = sent_tokenize(text)
    alpha_words = [w for w in words if w.isalpha()]

    features["word_count"] = len(alpha_words)
    features["unique_word_count"] = len(set(alpha_words))
    features["char_count"] = len(text)
    features["avg_word_length"] = np.mean([len(w) for w in alpha_words]) if alpha_words else 0
    features["sentence_count"] = len(sentences)
    features["avg_sentence_length"] = len(alpha_words) / max(len(sentences), 1)
    features["type_token_ratio"] = len(set(alpha_words)) / max(len(alpha_words), 1)

    content_words = [w for w in alpha_words if w not in STOP_WORDS]
    features["lexical_density"] = len(content_words) / max(len(alpha_words), 1)
    features["stopword_ratio"] = sum(1 for w in alpha_words if w in STOP_WORDS) / max(len(alpha_words), 1)
    features["long_word_ratio"] = sum(1 for w in alpha_words if len(w) > 6) / max(len(alpha_words), 1)

    tagged = pos_tag(alpha_words)
    pos_counts = Counter(tag for _, tag in tagged)
    total_tags = max(len(tagged), 1)

    features["noun_ratio"] = sum(pos_counts.get(t, 0) for t in ["NN","NNS","NNP","NNPS"]) / total_tags
    features["verb_ratio"] = sum(pos_counts.get(t, 0) for t in ["VB","VBD","VBG","VBN","VBP","VBZ"]) / total_tags
    features["adj_ratio"] = sum(pos_counts.get(t, 0) for t in ["JJ","JJR","JJS"]) / total_tags
    features["adv_ratio"] = sum(pos_counts.get(t, 0) for t in ["RB","RBR","RBS"]) / total_tags
    features["prep_ratio"] = pos_counts.get("IN", 0) / total_tags
    features["conj_ratio"] = sum(pos_counts.get(t, 0) for t in ["CC","IN"]) / total_tags
    features["pron_ratio"] = sum(pos_counts.get(t, 0) for t in ["PRP","PRP$","WP","WP$"]) / total_tags
    features["pos_diversity"] = len(pos_counts) / total_tags

    wl = [len(w) for w in alpha_words]
    features["word_length_std"] = np.std(wl) if len(wl) > 1 else 0

    word_freq = Counter(alpha_words)
    features["hapax_legomena_ratio"] = sum(1 for c in word_freq.values() if c == 1) / max(len(alpha_words), 1)

    def count_syllables(word):
        word = word.lower()
        count = 0
        vowels = "aeiou"
        if word[0] in vowels:
            count += 1
        for i in range(1, len(word)):
            if word[i] in vowels and word[i-1] not in vowels:
                count += 1
        if word.endswith("e"):
            count -= 1
        return max(count, 1)

    syllables = [count_syllables(w) for w in alpha_words if len(w) > 0]
    features["avg_syllables_per_word"] = np.mean(syllables) if syllables else 0
    features["complex_word_ratio"] = sum(1 for s in syllables if s >= 3) / max(len(alpha_words), 1)
    features["words_per_minute_approx"] = len(alpha_words) / (50 / 60)

    return features

train_ling_feats = pd.DataFrame([extract_linguistic_features(t) for t in train_df["transcript"]])
test_ling_feats = pd.DataFrame([extract_linguistic_features(t) for t in test_df["transcript"]])
print(f"Linguistic features: {train_ling_feats.shape}")

In [ ]:
tfidf_word = TfidfVectorizer(max_features=300, ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True, strip_accents="unicode")
tfidf_char = TfidfVectorizer(max_features=200, analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_df=0.95, sublinear_tf=True)

def clean_text(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-z\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

train_clean = [clean_text(t) for t in train_df["transcript"]]
test_clean = [clean_text(t) for t in test_df["transcript"]]

train_tfidf_word = tfidf_word.fit_transform(train_clean)
test_tfidf_word = tfidf_word.transform(test_clean)
train_tfidf_char = tfidf_char.fit_transform(train_clean)
test_tfidf_char = tfidf_char.transform(test_clean)

print(f"TF-IDF word: {train_tfidf_word.shape}, char: {train_tfidf_char.shape}")

In [ ]:
def extract_acoustic_features(filepath, sr=16000, duration=60):
    features = {}
    try:
        y, sr = librosa.load(filepath, sr=sr, duration=duration)
        features["audio_duration"] = len(y) / sr

        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        for i in range(13):
            features[f"mfcc_{i}_mean"] = np.mean(mfccs[i])
            features[f"mfcc_{i}_std"] = np.std(mfccs[i])
            features[f"mfcc_{i}_max"] = np.max(mfccs[i])
            features[f"mfcc_{i}_min"] = np.min(mfccs[i])

        delta_mfccs = librosa.feature.delta(mfccs)
        for i in range(13):
            features[f"delta_mfcc_{i}_mean"] = np.mean(delta_mfccs[i])
            features[f"delta_mfcc_{i}_std"] = np.std(delta_mfccs[i])

        sc = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
        features["spectral_centroid_mean"] = np.mean(sc)
        features["spectral_centroid_std"] = np.std(sc)

        sb = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
        features["spectral_bandwidth_mean"] = np.mean(sb)
        features["spectral_bandwidth_std"] = np.std(sb)

        sr_ = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
        features["spectral_rolloff_mean"] = np.mean(sr_)
        features["spectral_rolloff_std"] = np.std(sr_)

        scon = librosa.feature.spectral_contrast(y=y, sr=sr)
        for i in range(scon.shape[0]):
            features[f"spectral_contrast_{i}_mean"] = np.mean(scon[i])

        sf = librosa.feature.spectral_flatness(y=y)[0]
        features["spectral_flatness_mean"] = np.mean(sf)
        features["spectral_flatness_std"] = np.std(sf)

        zcr = librosa.feature.zero_crossing_rate(y)[0]
        features["zcr_mean"] = np.mean(zcr)
        features["zcr_std"] = np.std(zcr)

        rms = librosa.feature.rms(y=y)[0]
        features["rms_mean"] = np.mean(rms)
        features["rms_std"] = np.std(rms)
        features["rms_max"] = np.max(rms)

        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        for i in range(12):
            features[f"chroma_{i}_mean"] = np.mean(chroma[i])

        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        features["tempo"] = float(np.atleast_1d(tempo)[0])
        features["high_zcr_ratio"] = np.mean(zcr > np.mean(zcr))

        is_silent = rms < 0.01
        features["silence_ratio"] = np.mean(is_silent)
        features["num_pauses"] = np.sum(np.diff(is_silent.astype(int)) == 1)
    except Exception as e:
        return {}
    return features

print("Extracting acoustic features...")
train_acoustic_df = pd.DataFrame([extract_acoustic_features(get_audio_path(str(r[fname_col]), TRAIN_DIR)) for _, r in train_df.iterrows()])
test_acoustic_df = pd.DataFrame([extract_acoustic_features(get_audio_path(str(r[fname_col]), TEST_DIR)) for _, r in test_df.iterrows()])
print(f"Acoustic features: train={train_acoustic_df.shape}, test={test_acoustic_df.shape}")

## Combine Features & Train

In [ ]:
train_ling_feats = train_ling_feats.reset_index(drop=True)
test_ling_feats = test_ling_feats.reset_index(drop=True)
train_acoustic_df = train_acoustic_df.reset_index(drop=True)
test_acoustic_df = test_acoustic_df.reset_index(drop=True)

tw_df = pd.DataFrame(train_tfidf_word.toarray(), columns=[f"tw_{i}" for i in range(train_tfidf_word.shape[1])])
tw_test = pd.DataFrame(test_tfidf_word.toarray(), columns=[f"tw_{i}" for i in range(test_tfidf_word.shape[1])])
tc_df = pd.DataFrame(train_tfidf_char.toarray(), columns=[f"tc_{i}" for i in range(train_tfidf_char.shape[1])])
tc_test = pd.DataFrame(test_tfidf_char.toarray(), columns=[f"tc_{i}" for i in range(test_tfidf_char.shape[1])])

X_train = pd.concat([train_ling_feats, tw_df, tc_df, train_acoustic_df], axis=1)
X_test = pd.concat([test_ling_feats, tw_test, tc_test, test_acoustic_df], axis=1)
y_train = train_df[label_col].values.astype(float)

X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

common_cols = sorted(list(set(X_train.columns) & set(X_test.columns)))
X_train = X_train[common_cols]
X_test = X_test[common_cols]

print(f"Train: {X_train.shape}, Test: {X_test.shape}, Target range: [{y_train.min()}, {y_train.max()}]")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
models = {
    "Ridge": Ridge(alpha=10.0),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000),
    "SVR_RBF": SVR(kernel="rbf", C=5.0, epsilon=0.1),
    "SVR_Linear": SVR(kernel="linear", C=1.0, epsilon=0.1),
    "RandomForest": RandomForestRegressor(n_estimators=500, max_depth=12, min_samples_leaf=5, min_samples_split=10, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=500, max_depth=4, learning_rate=0.05, min_samples_leaf=5, subsample=0.8, random_state=42),
}
if xgb is not None:
    models["XGBoost"] = xgb.XGBRegressor(n_estimators=500, max_depth=4, learning_rate=0.05, min_child_weight=5, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbosity=0, n_jobs=-1)
if lgb is not None:
    models["LightGBM"] = lgb.LGBMRegressor(n_estimators=500, max_depth=5, learning_rate=0.05, num_leaves=31, min_child_samples=10, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbose=-1, n_jobs=-1)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}
cv_predictions = {}

for name, model in models.items():
    X_cv = X_train_scaled if name in ["Ridge","ElasticNet","SVR_RBF","SVR_Linear"] else X_train.values
    fold_preds = np.zeros(len(y_train))
    fold_metrics = {"rmse": [], "mae": [], "r2": [], "pearson": []}

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_cv)):
        m = type(model)(**model.get_params())
        m.fit(X_cv[tr_idx], y_train[tr_idx])
        preds = np.clip(m.predict(X_cv[val_idx]), 0, 5)
        fold_preds[val_idx] = preds
        y_val = y_train[val_idx]
        fold_metrics["rmse"].append(np.sqrt(mean_squared_error(y_val, preds)))
        fold_metrics["mae"].append(mean_absolute_error(y_val, preds))
        fold_metrics["r2"].append(r2_score(y_val, preds))
        pr, _ = pearsonr(y_val, preds)
        fold_metrics["pearson"].append(pr)

    cv_results[name] = {k: np.mean(v) for k, v in fold_metrics.items()}
    cv_predictions[name] = fold_preds

for name, res in cv_results.items():
    print(f"{name:<20} RMSE={res['rmse']:.4f}  Pearson={res['pearson']:.4f}")

best_model_name = max(cv_results, key=lambda k: cv_results[k]["pearson"])
print(f"\nBest: {best_model_name} (Pearson={cv_results[best_model_name]['pearson']:.4f})")

In [ ]:
sorted_models = sorted(cv_results.items(), key=lambda x: x[1]["pearson"], reverse=True)
top_model_names = [name for name, _ in sorted_models[:min(5, len(sorted_models))]]
estimators = [(name, type(models[name])(**models[name].get_params())) for name in top_model_names]

stack_preds = np.zeros(len(y_train))
stack_rmses, stack_pearsons = [], []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train_scaled)):
    sc = StackingRegressor(
        estimators=[(n, type(m)(**m.get_params())) for n, m in estimators],
        final_estimator=Ridge(alpha=1.0), cv=3, n_jobs=-1)
    sc.fit(X_train_scaled[tr_idx], y_train[tr_idx])
    preds = np.clip(sc.predict(X_train_scaled[val_idx]), 0, 5)
    stack_preds[val_idx] = preds
    stack_rmses.append(np.sqrt(mean_squared_error(y_train[val_idx], preds)))
    pr, _ = pearsonr(y_train[val_idx], preds)
    stack_pearsons.append(pr)

print(f"Stacking: RMSE={np.mean(stack_rmses):.4f}, Pearson={np.mean(stack_pearsons):.4f}")

In [ ]:
pearson_scores = {name: cv_results[name]["pearson"] for name in top_model_names}
total_pearson = sum(max(p, 0) for p in pearson_scores.values())

blend_oof = np.zeros(len(y_train))
for name in top_model_names:
    blend_oof += (max(pearson_scores[name], 0) / total_pearson) * cv_predictions[name]
blend_oof = np.clip(blend_oof, 0, 5)

blend_rmse = np.sqrt(mean_squared_error(y_train, blend_oof))
blend_pearson, _ = pearsonr(y_train, blend_oof)

print(f"Weighted Blend: RMSE={blend_rmse:.4f}, Pearson={blend_pearson:.4f}")

In [ ]:
best_single_pearson = cv_results[best_model_name]["pearson"]
stack_pearson = np.mean(stack_pearsons)

test_predictions = {}
for name in top_model_names:
    m = type(models[name])(**models[name].get_params())
    if name in ["Ridge","ElasticNet","SVR_RBF","SVR_Linear"]:
        m.fit(X_train_scaled, y_train)
        preds = m.predict(X_test_scaled)
    else:
        m.fit(X_train.values, y_train)
        preds = m.predict(X_test.values)
    test_predictions[name] = np.clip(preds, 0, 5)

final_preds_blend = np.zeros(len(X_test))
for name in top_model_names:
    final_preds_blend += (max(pearson_scores[name], 0) / total_pearson) * test_predictions[name]
final_preds_blend = np.clip(final_preds_blend, 0, 5)

stacking_final = StackingRegressor(
    estimators=[(n, type(m)(**m.get_params())) for n, m in estimators],
    final_estimator=Ridge(alpha=1.0), cv=5, n_jobs=-1)
stacking_final.fit(X_train_scaled, y_train)
final_preds_stack = np.clip(stacking_final.predict(X_test_scaled), 0, 5)

best_approach_pearson = max(best_single_pearson, stack_pearson, blend_pearson)
if best_approach_pearson == stack_pearson:
    final_predictions = final_preds_stack
else:
    final_predictions = final_preds_blend

print(f"Final predictions: mean={np.mean(final_predictions):.3f}, std={np.std(final_predictions):.3f}")

In [ ]:
submission = test_df[[fname_col]].copy()
submission[label_col] = final_predictions
submission.to_csv("submission.csv", index=False)
print(f"Saved submission.csv ({submission.shape})")
print(submission.head())

In [ ]:
sub_stack = test_df[[fname_col]].copy()
sub_stack[label_col] = final_preds_stack
sub_stack.to_csv("submission_stacking.csv", index=False)

sub_blend = test_df[[fname_col]].copy()
sub_blend[label_col] = final_preds_blend
sub_blend.to_csv("submission_blend.csv", index=False)